# P47

- Render attn pattern for all layers & heads

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import copy
import math
import time
from tqdm import tqdm
import colorsys
import jax

import torch
import torch.nn.functional as F

# openpi imports
from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config
from openpi.policies import aloha_policy
from openpi.shared import download
from openpi import transforms as _transforms
from openpi.shared import normalize as _normalize
from openpi.models import model as _model

from openpi.models_pytorch.pi0_pytorch import make_att_2d_masks

In [ ]:
joint_labels = [
    "L joint 0 (waist)", "L joint 1 (shoulder)", "L joint 2 (elbow)",
    "L joint 3 (forearm)", "L joint 4 (wrist angle)", "L joint 5 (wrist rotate)",
    "L gripper",
    "R joint 0 (waist)", "R joint 1 (shoulder)", "R joint 2 (elbow)",
    "R joint 3 (forearm)", "R joint 4 (wrist angle)", "R joint 5 (wrist rotate)",
    "R gripper",
]

In [ ]:
def boost_colors_hsv(colors, saturation_boost=1.0, min_saturation=0.0, min_value=0.0):
    """
    Adjust colors in HSV space for better visibility in the barcode viz.
 
    Parameters
    ----------
    colors : ndarray, shape (N, 3)
        RGB colors in [0, 1].
    saturation_boost : float
        Multiplier on S channel. 1.0 = no change, 1.5 = 50% more saturated.
    min_saturation : float in [0, 1]
        Floor for S channel. Prevents fully gray rows.
    min_value : float in [0, 1]
        Floor for V channel. Lifts dark/muddy colors so the hue reads clearly.
 
    Returns
    -------
    boosted : ndarray, shape (N, 3), dtype float32
    """
    colors = np.asarray(colors, dtype=np.float32)
    out = np.empty_like(colors)
    for i in range(len(colors)):
        h, s, v = colorsys.rgb_to_hsv(*colors[i])
        s = min(1.0, s * saturation_boost)
        s = max(s, min_saturation)
        v = max(v, min_value)
        out[i] = colorsys.hsv_to_rgb(h, s, v)
    return out
 
 
def _hsv_to_rgb_vec(hsv):
    """
    Vectorized HSV -> RGB.  hsv: (..., 3) with H in [0,1], S in [0,1], V in [0,1].
    Returns RGB (..., 3) in [0, 1].
    """
    h = hsv[..., 0]
    s = hsv[..., 1]
    v = hsv[..., 2]
 
    i = (h * 6.0).astype(np.int32) % 6
    f = (h * 6.0) - np.floor(h * 6.0)
    p = v * (1.0 - s)
    q = v * (1.0 - s * f)
    t = v * (1.0 - s * (1.0 - f))
 
    out = np.empty_like(hsv)
    for idx, (r, g, b) in enumerate([(v,t,p), (q,v,p), (p,v,t), (p,q,v), (t,p,v), (v,p,q)]):
        mask = (i == idx)
        out[..., 0] = np.where(mask, r, out[..., 0])
        out[..., 1] = np.where(mask, g, out[..., 1])
        out[..., 2] = np.where(mask, b, out[..., 2])
    return out
 
 
def render_custom_rows(embeddings, colors, padding=0.1, aspect_ratio=(1, 2), brightness_range=0.85,
                       saturation_boost=1.0, min_saturation=0.0, min_value=0.0,
                       cmap_mode='brightness'):
    """
    Render embedding vectors as colored rows with custom per-row colormaps.
 
    Parameters
    ----------
    embeddings : ndarray, shape (N, D)
        N embedding vectors of dimension D.
    colors : ndarray, shape (N, 3)
        RGB color for each row, values in [0, 1].
    padding : float in [0, 1]
        Gap between rows as a fraction of the base row height (10px).
        e.g. padding=0.1 -> 1px gap, padding=0.5 -> 5px gap.
    aspect_ratio : tuple (H, W)
        Desired aspect ratio of the output image.
    brightness_range : float in (0, 1]
        How far toward black/white the colormap extends from the base color.
        1.0 = full black-to-color-to-white. 0.5 = more subtle variation.
    saturation_boost : float
        Multiplier on HSV saturation. >1 pumps up color intensity.
    min_saturation : float in [0, 1]
        Floor for saturation. 0.3 guarantees nothing looks fully gray.
    min_value : float in [0, 1]
        Floor for HSV value. Lifts dark/muddy colors so hue reads clearly.
        e.g. 0.5 ensures no row's base color is darker than mid-gray.
    cmap_mode : str, 'brightness' or 'hsv'
        'brightness': dark <-> base color <-> white (in RGB space).
        'hsv': modulates S and V around the base color's hue.
               Low embedding values -> desaturated & darker (same hue).
               High embedding values -> more saturated & brighter (same hue).
 
    Returns
    -------
    img : ndarray, shape (total_h, total_w, 3), dtype float32
        RGB image array ready for imshow().
    """
    embeddings = np.asarray(embeddings, dtype=np.float32)
    colors = np.asarray(colors, dtype=np.float32)
 
    # Support single-row input: (D,) -> (1, D), (3,) -> (1, 3)
    if embeddings.ndim == 1:
        embeddings = embeddings[None, :]
    if colors.ndim == 1:
        colors = colors[None, :]
 
    N, D = embeddings.shape
 
    # --- NaN colors -> black ---
    nan_mask = np.any(np.isnan(colors), axis=1)  # (N,)
    if nan_mask.any():
        colors = colors.copy()
        colors[nan_mask] = 0.0  # safe placeholder for boost
 
    # --- Boost colors in HSV space ---
    if saturation_boost != 1.0 or min_saturation > 0.0 or min_value > 0.0:
        colors = boost_colors_hsv(colors, saturation_boost, min_saturation, min_value)
 
    # Re-zero NaN rows so min_value doesn't lift them
    if nan_mask.any():
        colors[nan_mask] = 0.0
 
    # --- Geometry ---
    row_h = 10
    gap_h = max(1, round(padding * row_h)) if padding > 0 else 0
 
    total_h = N * row_h + max(0, N - 1) * gap_h
    total_w = int(total_h * aspect_ratio[1] / aspect_ratio[0])
    total_w = max(total_w, 1)
 
    img = np.zeros((total_h, total_w, 3), dtype=np.float32)
 
    # --- Precompute resampling coordinates ---
    x_orig = np.linspace(0, 1, D)
    x_new = np.linspace(0, 1, total_w)
 
    for i in range(N):
        # NaN color rows stay black (img is zero-initialized)
        if nan_mask.any() and nan_mask[i]:
            continue
 
        row = embeddings[i]
 
        # Normalize to [0, 1] per row
        lo, hi = row.min(), row.max()
        if hi - lo > 1e-12:
            norm = (row - lo) / (hi - lo)
        else:
            norm = np.full_like(row, 0.5)
 
        # Resample from D -> total_w via linear interpolation
        resampled = np.interp(x_new, x_orig, norm)  # (total_w,)
 
        # --- Per-row colormap ---
        c = colors[i]  # (3,)
 
        if cmap_mode == 'brightness':
            # dark <-> color <-> white in RGB space
            dark_end = c * (1.0 - brightness_range)
            bright_end = c + (1.0 - c) * brightness_range
 
            pixels = np.empty((total_w, 3), dtype=np.float32)
            mask_lo = resampled <= 0.5
            mask_hi = ~mask_lo
 
            t_lo = (resampled[mask_lo] * 2.0)[:, None]
            pixels[mask_lo] = dark_end + t_lo * (c - dark_end)
 
            t_hi = ((resampled[mask_hi] - 0.5) * 2.0)[:, None]
            pixels[mask_hi] = c + t_hi * (bright_end - c)
 
        elif cmap_mode == 'hsv':
            # Modulate S and V around the base color's hue
            h_base, s_base, v_base = colorsys.rgb_to_hsv(*c)
 
            s_lo = s_base * (1.0 - brightness_range)
            v_lo = v_base * (1.0 - brightness_range)
            s_hi = s_base + (1.0 - s_base) * brightness_range
            v_hi = v_base + (1.0 - v_base) * brightness_range
 
            # Two-piece linear interp in HSV, same structure as brightness mode
            hsv_pixels = np.empty((total_w, 3), dtype=np.float32)
            hsv_pixels[:, 0] = h_base  # hue is constant
 
            mask_lo = resampled <= 0.5
            mask_hi = ~mask_lo
 
            t_lo = resampled[mask_lo] * 2.0      # 0->0, 0.5->1
            hsv_pixels[mask_lo, 1] = s_lo + t_lo * (s_base - s_lo)
            hsv_pixels[mask_lo, 2] = v_lo + t_lo * (v_base - v_lo)
 
            t_hi = (resampled[mask_hi] - 0.5) * 2.0  # 0.5->0, 1->1
            hsv_pixels[mask_hi, 1] = s_base + t_hi * (s_hi - s_base)
            hsv_pixels[mask_hi, 2] = v_base + t_hi * (v_hi - v_base)
 
            pixels = _hsv_to_rgb_vec(hsv_pixels)
 
        else:
            raise ValueError(f"Unknown cmap_mode: {cmap_mode!r}")
 
        # Paint into row band
        y0 = i * (row_h + gap_h)
        img[y0 : y0 + row_h] = pixels[None, :, :]  # broadcast across row height
 
    return np.clip(img, 0.0, 1.0)

In [ ]:
def frame_to_obs(frame):
    """Convert a HuggingFace dataset frame to the observation dict
    that the policy expects (post-RepackTransform format)."""
    def to_chw(img):
        arr = np.array(img)  # PIL -> HWC (H, W, 3)
        return arr.transpose(2, 0, 1)  # -> CHW (3, H, W)
    
    return {
        "images": {
            "cam_high": to_chw(frame["observation.images.cam_high"]),
            "cam_left_wrist": to_chw(frame["observation.images.cam_left_wrist"]),
            "cam_right_wrist": to_chw(frame["observation.images.cam_right_wrist"]),
        },
        "state": np.array(frame["observation.state"]),
        "prompt": "uncap the pen",
    }

## 1. Load Dataset

In [ ]:
ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")
print(f"Total rows: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"Episode indices: {sorted(set(ds['episode_index']))}")

In [ ]:
# Pick an episode and grab all its frames
EPISODE_IDX = 0

ep_indices = np.array(ds["episode_index"])
mask = ep_indices == EPISODE_IDX
row_indices = np.where(mask)[0]
episode = ds.select(row_indices)

In [ ]:
# Pick a frame in the middle of the episode
FRAME_IDX = 150
frame = episode[FRAME_IDX]

In [ ]:
# Display camera views
cam_names = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, cam_names):
    img = frame[f"observation.images.{name}"]
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle(f"Episode {EPISODE_IDX}, Frame {FRAME_IDX} (t={frame['timestamp']:.2f}s)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Load PyTorch Model

In [ ]:
config = _config.get_config("pi0_aloha_pen_uncap")

# Point to converted PyTorch checkpoint
checkpoint_dir = "data/robots/openpi/pi0_aloha_pen_uncap_pytorch"

print(f"Model type: {config.model.model_type}")
print(f"Action dim: {config.model.action_dim}")
print(f"Action horizon: {config.model.action_horizon}")

In [ ]:
# Create the trained policy — auto-detects PyTorch format
policy = _policy_config.create_trained_policy(config, checkpoint_dir)
print(f"Policy created successfully!")
print(f"PyTorch model: {policy._is_pytorch_model}")
print(f"Device: {policy._pytorch_device}")
print(f"Number of input transforms: {len(policy._input_transform.transforms)}")
print(f"Number of output transforms: {len(policy._output_transform.transforms)}")

In [ ]:
# Quick reference inference to make sure the model works
obs_test = frame_to_obs(frame)
result_test = policy.infer(obs_test)
print(f"Actions shape: {result_test['actions'].shape}")
print(f"Inference time: {result_test['policy_timing']['infer_ms']:.1f} ms")

## 3. Reference Inference with Fixed Noise

Generate fixed noise (using numpy so we don't need JAX at all),
run `policy.infer` as a black box, and save the result as our reference.

In [ ]:
# Fix noise for reproducibility (numpy, no JAX dependency)
rng = np.random.RandomState(42)
fixed_noise = rng.randn(50, 32).astype(np.float32)
print(f"fixed_noise: {fixed_noise.shape}, dtype={fixed_noise.dtype}")
print(f"  norm={np.linalg.norm(fixed_noise):.4f}, mean={fixed_noise.mean():.4f}")

# Reference result using policy.infer as a black box
obs_ref = frame_to_obs(frame)
result_ref = policy.infer(obs_ref, noise=fixed_noise)
print(f"\nReference actions: {result_ref['actions'].shape}")
print(f"  range: [{result_ref['actions'].min():.4f}, {result_ref['actions'].max():.4f}]")

## 4. Manual Reproduction — Input Transforms

Replicate what `policy.infer()` does step by step.

In [ ]:
device = torch.device(policy._pytorch_device)
pt_model = policy._model  # the PI0Pytorch instance

obs_manual = frame_to_obs(frame)
inputs = policy._input_transform(obs_manual)

inputs_pt = jax.tree.map(
    lambda x: torch.from_numpy(np.array(x)).to(device)[None, ...],
    inputs
)

observation = _model.Observation.from_dict(inputs_pt)

images, img_masks, lang_tokens, lang_masks, state = pt_model._preprocess_observation(
    observation, train=False
)

with torch.no_grad():
    prefix_embs, prefix_pad_masks, prefix_att_masks = pt_model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks
    )

with torch.no_grad():
    embs_list = []
    pad_mask_list = []
    ar_mask_list = []

    # --- Encode each image through SigLIP ---
    # In PyTorch: paligemma.model.get_image_features
    # This is HuggingFace's PaliGemma vision tower (SigLIP So400m/14)
    for i, (img, img_mask) in enumerate(zip(images, img_masks)):
        img_emb = pt_model.paligemma_with_expert.embed_image(img)
        bsize, num_img_tokens = img_emb.shape[:2]

        print(f"SigLIP image {i}:")
        print(f"  input:  {img.shape}, dtype={img.dtype}, range=[{img.min():.2f}, {img.max():.2f}]")
        print(f"  output: {img_emb.shape}, dtype={img_emb.dtype}")

        embs_list.append(img_emb)
        pad_mask_list.append(img_mask[:, None].expand(bsize, num_img_tokens))
        ar_mask_list += [0] * num_img_tokens

    # --- Embed language tokens ---
    # PaliGemma's embedding table + scale by sqrt(dim)
    lang_emb = pt_model.paligemma_with_expert.embed_language_tokens(lang_tokens)
    lang_emb_dim = lang_emb.shape[-1]
    lang_emb = lang_emb * math.sqrt(lang_emb_dim)

    print(f"\nLanguage embedding:")
    print(f"  token IDs: {lang_tokens.shape}, non-padding: {lang_masks.sum()}")
    print(f"  embeddings: {lang_emb.shape}, dtype={lang_emb.dtype}")

    embs_list.append(lang_emb)
    pad_mask_list.append(lang_masks)
    ar_mask_list += [0] * lang_emb.shape[1]

    # --- Concatenate into full prefix sequence ---
    prefix_embs_manual = torch.cat(embs_list, dim=1)
    prefix_pad_masks_manual = torch.cat(pad_mask_list, dim=1)

    print(f"\nFull prefix:")
    print(f"  embs:      {prefix_embs_manual.shape}  (3x256 image + 48 language = 816)")
    print(f"  pad_masks: {prefix_pad_masks_manual.shape}  (valid: {prefix_pad_masks_manual.sum()})")

    # --- Compare against reference ---
    max_diff = (prefix_embs_manual - prefix_embs).abs().max().item()
    print(f"\nMax diff vs embed_prefix reference: {max_diff}")



with torch.no_grad():
    # Build attention mask
    prefix_att_2d = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_positions = torch.cumsum(prefix_pad_masks, dim=1) - 1

    print(f"prefix_att_2d: {prefix_att_2d.shape}")  # [1, 816, 816]
    print(f"prefix_positions: {prefix_positions.shape}")  # [1, 816]

    # Convert to 4D attention mask (the format HuggingFace expects)
    prefix_att_4d = pt_model._prepare_attention_masks_4d(prefix_att_2d)
    print(f"prefix_att_4d: {prefix_att_4d.shape}")  # [1, 1, 816, 816]

    # Set eager attention (not SDPA) so we can hook into it
    pt_model.paligemma_with_expert.paligemma.language_model.config._attn_implementation = "eager"

    # Forward pass: [prefix_embs, None] — only PaliGemma runs, action expert gets None
    _, kv_cache = pt_model.paligemma_with_expert.forward(
        attention_mask=prefix_att_4d,
        position_ids=prefix_positions,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

print(f"\nKV cache type: {type(kv_cache).__name__}")
print(f"KV cache layers: {len(kv_cache)}")
# Each layer has (key, value) tensors
k0, v0 = kv_cache[0]
print(f"Layer 0 K: {k0.shape}, V: {v0.shape}")  # [1, num_kv_heads, 816, head_dim]

### Pre Softmax

In [ ]:
attn_logits = {}
attn_weights = {}
hooks = []

def make_deep_attn_hook(layer_idx):
    original_forward = None
    
    def patched_forward(self, *args, **kwargs):
        # Run the normal forward
        kwargs['output_attentions'] = True
        result = original_forward(*args, **kwargs)
        
        # Compute logits manually from Q, K
        # We can get them by hooking the softmax input instead
        return result
    
    # Simpler approach: hook the attention dropout or softmax
    def hook_fn(module, inp, out):
        # For GemmaAttention, we can compute logits from Q and K
        # But easier: just capture both scales
        if isinstance(out, tuple) and out[1] is not None:
            attn_weights[layer_idx] = out[1].detach().cpu()
    return hook_fn

# Actually, the cleanest approach: compute logits from Q and K directly
q_captures = {}
k_captures = {}
v_captures = {}

def make_q_hook(layer_idx):
    def hook_fn(module, inp, out):
        q_captures[layer_idx] = out.detach().cpu()
    return hook_fn

def make_k_hook(layer_idx):
    def hook_fn(module, inp, out):
        k_captures[layer_idx] = out.detach().cpu()
    return hook_fn

def make_v_hook(layer_idx):
    def hook_fn(module, inp, out):
        v_captures[layer_idx] = out.detach().cpu()
    return hook_fn

pali_layers = pt_model.paligemma_with_expert.paligemma.language_model.layers
for i, layer in enumerate(pali_layers):
    hooks.append(layer.self_attn.q_proj.register_forward_hook(make_q_hook(i)))
    hooks.append(layer.self_attn.k_proj.register_forward_hook(make_k_hook(i)))
    hooks.append(layer.self_attn.v_proj.register_forward_hook(make_v_hook(i)))

with torch.no_grad():
    _, kv_cache = pt_model.paligemma_with_expert.forward(
        attention_mask=prefix_att_4d,
        position_ids=prefix_positions,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

for h in hooks:
    h.remove()

#Attention patterns
pali_attn_patterns = {}
hooks = []

def make_attn_hook(layer_idx):
    def hook_fn(module, inp, output):
        # GemmaAttention.forward returns (attn_output, attn_weights, ...)
        if isinstance(output, tuple) and len(output) >= 2 and output[1] is not None:
            # attn_weights shape: [batch, num_heads, seq_len, seq_len]
            pali_attn_patterns[layer_idx] = output[1].detach().cpu()
    return hook_fn

pali_layers = pt_model.paligemma_with_expert.paligemma.language_model.layers
for i, layer in enumerate(pali_layers):
    h = layer.self_attn.register_forward_hook(make_attn_hook(i))
    hooks.append(h)

print(f"Hooked {len(hooks)} attention layers")

# Run the prefix forward pass
with torch.no_grad():
    _, kv_cache = pt_model.paligemma_with_expert.forward(
        attention_mask=prefix_att_4d,
        position_ids=prefix_positions,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

# Clean up hooks
for h in hooks:
    h.remove()

In [ ]:
queries=q_captures[8][0, :, 6*256:7*256].detach().cpu().float().numpy() #Should be attention head 6!
keys=k_captures[8][0, :, :].detach().cpu().float().numpy() #Should be attention head 6!
values=v_captures[8][0, :, :].detach().cpu().float().numpy() #Should be attention head 6!

In [ ]:
attn_pattern=pali_attn_patterns[8][0, 6].cpu().float().numpy()

In [ ]:
attn_scores=np.dot(queries, keys.T) #Row or column from this as heatmap is a good sanity check

In [ ]:
my_attn_pattern=torch.nn.Softmax(1)(torch.tensor(attn_scores)/16).numpy() #Quick hack but visuall similar Doesn't inlcude mask or Rope

In [ ]:
attn_scores.shape, my_attn_pattern.shape

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cm as cm

hot_full = cm.get_cmap('hot')
hot_truncated = LinearSegmentedColormap.from_list(
    'hot_trunc', hot_full(np.linspace(0, 0.65, 256))
)

magenta_cmap_bk = LinearSegmentedColormap.from_list(
    'magenta_hot', 
    ['black', 'darkmagenta', 'magenta']
)

magenta_cmap = LinearSegmentedColormap.from_list(
    'magenta_hot',
    [(1, 1, 1, 0), (0.5, 0, 0.5, 0.8), (1, 0, 1, 1)]
)

In [ ]:
# plt.imshow(attn_pattern, cmap=magenta_cmap_bk,
#                        vmin=torch.tensor(attn_pattern).quantile(0.01).item(), 
#                        vmax=torch.tensor(attn_pattern).quantile(0.99).item())

In [ ]:
attn_pattern=pali_attn_patterns[layer][0, head].cpu().float().numpy()

fig=plt.figure(0, (1,1), facecolor='k')
ax=fig.add_subplot(111)
plt.imshow(attn_pattern[:772, :772], cmap=magenta_cmap_bk,
                       vmin=torch.tensor(attn_pattern[:772, :772]).quantile(0.001).item(), 
                       vmax=torch.tensor(attn_pattern[:772, :772]).quantile(0.8).item())
plt.axis('off')

In [ ]:
save_dir='data/robots/p47'

import os
os.makedirs(save_dir, exist_ok=True)

In [ ]:
for layer in tqdm(range(18)):
    for head in range(8):
        plt.clf()
        attn_pattern=pali_attn_patterns[layer][0, head].cpu().float().numpy()
        
        fig=plt.figure(0, (4,4), facecolor='k')
        ax=fig.add_subplot(111)
        plt.imshow(attn_pattern[:772, :772], cmap=magenta_cmap_bk,
                               vmin=torch.tensor(attn_pattern[:772, :772]).quantile(0.001).item(), 
                               vmax=torch.tensor(attn_pattern[:772, :772]).quantile(0.8).item())
        plt.axis('off')
        plt.savefig(save_dir+'/attn_pattern_'+str(layer)+'_'+str(head)+'.png', transparent=True, dpi=100)